In [ ]:
#| default_exp compute

In [ ]:
#| include: false
from nbdev.showdoc import *

In [ ]:
#| export
from __future__ import annotations

import math
import warnings
from dataclasses import dataclass

import torch
import torch.nn as nn

from fasterbench.core import _is_quantized

try:
    from thop import profile as _thop_profile
except ImportError:
    _thop_profile = None

try:
    from torchprofile import profile_macs as _profile_macs
except ImportError:
    _profile_macs = None

In [ ]:
#| export
@dataclass(slots=True)
class ComputeMetrics:
    """MACs (Multiply-Accumulate operations) in millions."""
    macs_m: float | None  # None if unavailable

    @property
    def macs_available(self) -> bool:
        """Check if MACs measurement succeeded."""
        return self.macs_m is not None

    def as_dict(self) -> dict[str, float]:
        return {
            "macs_m": self.macs_m if self.macs_m is not None else float("nan"),
        }


#| export
def _count_macs_hooks(
    model: nn.Module,      # model to profile (run on CPU)
    sample: torch.Tensor,  # input tensor (with batch dimension)
) -> int:
    """Count Conv2d/Linear MACs via forward hooks (fp *and* quantized modules).

    thop has no handlers for quantized ops (it reports ~0), so we count manually.
    Quantization does not change the number of multiply-accumulates: an int8 conv
    does the same MACs as its fp32 twin. Uses module attributes (`out_channels`,
    `kernel_size`, `in_features`, ...) which both fp and quantized modules expose,
    and reads output spatial dims from the produced tensor.
    """
    macs = 0
    handles = []

    def conv_hook(m, inp, out):
        nonlocal macs
        out_spatial = math.prod(out.shape[2:])                       # out_H * out_W
        kernel = math.prod(m.kernel_size)                            # kH * kW
        macs += out.shape[0] * m.out_channels * (m.in_channels // m.groups) * kernel * out_spatial

    def linear_hook(m, inp, out):
        nonlocal macs
        leading = out.numel() // m.out_features                      # batch/token dims
        macs += leading * m.in_features * m.out_features

    for m in model.modules():
        if hasattr(m, "kernel_size") and hasattr(m, "out_channels") and hasattr(m, "groups"):
            handles.append(m.register_forward_hook(conv_hook))
        elif hasattr(m, "in_features") and hasattr(m, "out_features"):
            handles.append(m.register_forward_hook(linear_hook))

    try:
        with torch.no_grad():
            model(sample)
    finally:
        for h in handles:
            h.remove()
    return macs


#| export
def compute_compute(
    model: nn.Module,        # model to analyze
    sample: torch.Tensor,    # input tensor (with batch dimension)
) -> ComputeMetrics:
    """Compute MACs for a single forward pass."""
    try:
        model_device = next(model.parameters()).device
    except StopIteration:
        model_device = torch.device("cpu")

    if sample.device != model_device:
        sample = sample.to(model_device)

    macs_m: float | None = None

    # Quantized models: thop has no quantized-op handlers (returns ~0). Count the
    # Conv2d/Linear MACs manually — quantization preserves the MAC count.
    if _is_quantized(model):
        try:
            macs_m = round(_count_macs_hooks(model, sample) / 1e6, 3)
        except Exception as e:
            warnings.warn(f"quantized MAC counting failed: {e}")
        return ComputeMetrics(macs_m=macs_m)

    if _thop_profile is not None:
        try:
            mac_raw, _ = _thop_profile(model, inputs=(sample,), verbose=False)
            macs_m = round(mac_raw / 1e6, 3)
        except Exception as e:
            warnings.warn(f"thop failed: {e}")
    elif _profile_macs is not None:
        try:
            macs_m = round(_profile_macs(model, sample) / 1e6, 3)
        except Exception as e:
            warnings.warn(f"torchprofile failed: {e}")
    else:
        warnings.warn("No MAC-counting backend available – skipping MACs")

    return ComputeMetrics(macs_m=macs_m)

In [ ]:
show_doc(ComputeMetrics)

In [ ]:
show_doc(compute_compute)

In [ ]:
#| hide
from fastcore.test import *

import torch, torch.nn as nn
_m = nn.Linear(10, 5)
_x = torch.randn(1, 10)
_c = compute_compute(_m, _x)
assert isinstance(_c, ComputeMetrics)

In [ ]:
#| hide
# Quantization preserves the *number* of MACs (int8 conv/linear do the same
# multiply-accumulates as their fp32 twin) — only the ops get cheaper. But thop
# has no quantized-op handlers, so it reports ~0. Verify the manual hook counter
# recovers a MAC count ~equal to the fp32 twin's thop count.
from torch.ao.quantization import get_default_qconfig, QConfigMapping
from torch.ao.quantization.quantize_fx import prepare_fx, convert_fx
from fasterbench.core import _is_quantized

class _MacNet(nn.Module):
    "Conv-dominated net (linear/pool MACs negligible), statically quantizable via FX."
    def __init__(self):
        super().__init__()
        self.c1, self.c2 = nn.Conv2d(3, 16, 3, padding=1), nn.Conv2d(16, 32, 3, padding=1)
        self.pool, self.flat, self.fc = nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(32, 10)
    def forward(self, x):
        x = torch.relu(self.c1(x)); x = torch.relu(self.c2(x))
        return self.fc(self.flat(self.pool(x)))

_fp = _MacNet().eval()
_x = torch.randn(1, 3, 16, 16)
_fp_macs = compute_compute(_fp, _x).macs_m          # fp32 baseline (thop path)
assert _fp_macs is not None and _fp_macs > 0

torch.backends.quantized.engine = "x86"
_qmap = QConfigMapping().set_global(get_default_qconfig("x86"))
_prep = prepare_fx(_MacNet().eval(), _qmap, example_inputs=(_x,))
_prep(_x)                                           # calibrate
_qm = convert_fx(_prep)
assert _is_quantized(_qm)

# BEFORE the fix, thop reports ~0 MACs for a fully quantized model.
_thop_raw, _ = _thop_profile(_qm, inputs=(_x,), verbose=False)
assert _thop_raw / 1e6 < 0.1 * _fp_macs             # thop is effectively blind here

_q = compute_compute(_qm, _x)                       # AFTER the fix (manual counter)
assert _q.macs_m is not None and _q.macs_m > 0      # zero is gone
assert abs(_q.macs_m - _fp_macs) <= 0.05 * _fp_macs # within ~5% of fp32 MACs

---

## See Also

- [Size](size.html) — Model size measurement
- [Benchmark](../analysis/benchmark.html) — Unified API